In [40]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from pathlib import Path
from typing import List, Dict, Any, Optional
from tqdm.auto import tqdm
from pypdf import PdfReader
from dotenv import load_dotenv

# Langhchain framework
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# HF + Groq clients
from huggingface_hub import InferenceClient
from groq import Groq

import warnings
warnings.filterwarnings("ignore")

## Load API keys for credential key

In [41]:
ENV_PATH = Path.cwd().parent.parent / "credit_risk_production" / ".env"
load_dotenv(dotenv_path=ENV_PATH)

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(f"HF token loaded:  {HUGGINGFACE_API_KEY[:3]}...")
print(f"GROQ token loaded: {GROQ_API_KEY[:3]}...")

HF token loaded:  hf_...
GROQ token loaded: gsk...


## Load database

In [42]:
DATA = Path.cwd().parent.parent / "credit_risk_production" / "database" / "data" / "merged_credit_risk_data_with_fraud.parquet"
FEATURES = Path.cwd().parent.parent / "credit_risk_production" / "database" / "data" / "features_data.parquet"

df = pd.read_parquet(DATA)
features = pd.read_parquet(FEATURES)

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Features loaded: {features.shape[0]} rows, {features.shape[1]} columns")

Data loaded: 51336 rows, 88 columns
Features loaded: 51336 rows, 47 columns


## Load Fraud ML Models

In [43]:
# Load ML models from bundle
MODEL_BUNDLE = Path.cwd().parent.parent / "data_science" / "models" / "fraud_models" / "model_bundle.joblib"
with open(MODEL_BUNDLE, "rb") as f:
    model_bundle = joblib.load(f)
    print(f"Model bundle loaded: {list(model_bundle.keys())}")

# Load best parameters for each model
PARAMS = Path.cwd().parent.parent / "data_science" / "models" / "metrics" / "best_parameters.json"
with open(PARAMS, "r") as f:
    best_params = json.load(f)
    print(f"Best parameters loaded: {list(best_params.keys())}")

# Load metrics model
METRICS = Path.cwd().parent.parent / "data_science" / "models" / "metrics" / "model_metrics.csv"
metrics_df = pd.read_csv(METRICS)
display(metrics_df)

Model bundle loaded: ['models', 'scaler', 'label_encoders', 'feature_columns', 'class_labels']
Best parameters loaded: ['Logistic Regression', 'Random Forest', 'Decision Tree', 'XGBoost', 'K-Nearest Neighbors']


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.957148,0.955984,0.988137,0.971795,0.958286
1,Random Forest,0.982859,0.986372,0.990744,0.988554,0.998319
2,Decision Tree,0.986560,0.995266,0.986703,0.990966,0.993473
3,XGBoost,0.987924,0.998547,0.985269,0.991864,0.998564
4,K-Nearest Neighbors,0.869108,0.885370,0.947464,0.915365,0.906852


In [44]:
model_bundle['models']

{'Logistic Regression': LogisticRegression(C=10, random_state=42),
 'Random Forest': RandomForestClassifier(bootstrap=False, min_samples_leaf=2,
                        min_samples_split=10, random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=4, min_samples_split=5,
                        random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=1.0, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               gamma=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.01, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=5, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
        

In [45]:
best_params

{'Logistic Regression': {'solver': 'lbfgs', 'penalty': 'l2', 'C': 10},
 'Random Forest': {'n_estimators': 100,
  'min_samples_split': 10,
  'min_samples_leaf': 2,
  'max_depth': None,
  'bootstrap': False},
 'Decision Tree': {'min_samples_split': 5,
  'min_samples_leaf': 4,
  'max_depth': 10},
 'XGBoost': {'n_estimators': 200,
  'max_depth': 5,
  'learning_rate': 0.01,
  'colsample_bytree': 1.0},
 'K-Nearest Neighbors': {'weights': 'distance',
  'n_neighbors': 9,
  'metric': 'manhattan'}}

# 📚 Document Ingestion — PDF → Chunks (per-document strategy)

In [46]:
PDF_DIR = Path("../../credit_risk_production/database/pdf")

pdf_files = {
    "delinquency":  PDF_DIR / "Delinquency_Classification .pdf",
    "fraud":        PDF_DIR / "Fraud_Typologies_and_Red Flags .pdf",
    "regulatory":   PDF_DIR / "Regulatory_Risk Policy Core .pdf",
    "scorecard":    PDF_DIR / "Scorecard_Cut-off Policy.pdf"
}

raw_texts: Dict[str, str] = {}

for name, path in pdf_files.items():
    reader = PdfReader(str(path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    raw_texts[name] = text
    print(f"{name:12s} | pages={len(reader.pages):3d} | chars={len(text):,}")

delinquency  | pages= 93 | chars=191,238
fraud        | pages=  9 | chars=25,113
regulatory   | pages= 11 | chars=34,419
scorecard    | pages=178 | chars=465,508


## Chunking strategy to retrieve Document augmentation for each PDFs to enrich vocab on LLM
- #### Strategy 1: Delinquency Classification — rule-based chunking
- #### Strategy 2: Fraud Typologies — focusing on fraud paragraph to be chunked
- #### Strategy 3: Regulatory Policy — recursive with heading preservation
- #### Strategy 4: Scorecard Cut-off — table-aware chunking
- #### Final Strategy: Combine all chunks

In [47]:
def chunk_delinquency(text: str) -> List[Document]:
    """Split by classification headings, falls back to paragraph split."""
    keywords = ["Standard", "Sub-standard", "Substandrad", "Doubtful", "Loss"]

    # Crude split: find the index of each keyword and slice
    positions = []
    for kw in keywords:
        idx = text.lower().find(kw.lower())
        if idx != -1:
            positions.append((idx, kw))
    positions.sort()

    docs: List[Document] = []
    if not positions:

        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "delinquency", "chunk_id": i, "class_level": "unknown"}
            ))
        return docs

    for i, (start, kw) in enumerate(positions):
        end = positions[i + 1][0] if i + 1 < len(positions) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue

        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "delinquency", "class_level": kw.lower(), "chunk_id": i},
        ))

    return docs

# Usage on delinquency PDF
deling_docs = chunk_delinquency(raw_texts["delinquency"])
print(f"Delinquency chunks: {len(deling_docs)}")
print(deling_docs[3].page_content[:300], "\n--")

Delinquency chunks: 4
sub-standard' immediately on restructuring, 
all borrowers, with the exception of the borrowal categories specified in para 14.1 below ( i.e 
consumer and personal advances, advances classified as capital market and real estate 
exposures), will be entitled to retain the asset classification upon re 
--


#### Strategy 2: Fraud Typologies — focusing on fraud paragraph to be chunked

In [48]:
def chunk_fraud(text: str) -> List[Document]:
    """Split by typology headings, falls back to paragraph split."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=80,
        separators=["\n\n\n", "\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "fraud", "chunk_id": i, "typology": "unspecified"}
        ))
    return docs

# Usage on fraud PDF
fraud_docs = chunk_fraud(raw_texts["fraud"])
print(f"Fraud chunks: {len(fraud_docs)}")
print(fraud_docs[3].page_content[:300], "\n--")

Fraud chunks: 36
platforms achieved a 61% improvement in detection accuracy, 48% reduction in false positives, and 72% faster 
investigation turnaround. The project management model introduced in this article outlines the lifecycle for planning, 
building, validating, deploying, and governing these systems, emphasiz 
--


### Strategy 3: Regulatory Policy — recursive with heading preservation

In [49]:
def chunk_regulatory(text: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "regulatory", "chunk_id": i},
        ))
    return docs

# Usage on regulatory PDF
regulatory_docs = chunk_regulatory(raw_texts["regulatory"])
print(f"Regulatory chunks: {len(regulatory_docs)}")
print(regulatory_docs[1].page_content[:411], "\n--")

Regulatory chunks: 53
(FPC). However, despite these guidelines, rising consumer complaints indicate a gap between regulatory 
expectations and actual practices. This study aims to empirically examine the compliance of FPC norms among 
Banks and Housing Finance Companies (HFCs), as perceived by lending officials and borrowers. Primary data 
were collected from 294 borrowers and 102 lending branches using structured questionnaires. 
--


### Strategy 4: Scorecard Cut-off — table-aware chunking

In [50]:
import re

def chunk_scorecard(text: str) -> List[Document]:
    """Detect numeric ranges like 700-750 or 700 - 750 and split accordingly."""
    pattern = re.compile(r"(\d{3})\s*[--to]+\s*(\d{3})")
    matches = list(pattern.finditer(text))

    docs: List[Document] = []
    if not matches:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=700, 
            chunk_overlap=80,
            separators=["\n\n", "\n", ". ", " "]
        )
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "scorecard", "chunk_id": i, "min_score": None, "max_score": None}
            ))
        return docs

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue
        docs.append(Document(
            page_content=chunk,
            metadata={
                "doc_type": "scorecard",
                "chunk_id": i,
                "min_score": int(m.group(1)),
                "max_score": int(m.group(2))
            }
        ))
    return docs

# Usage on scorecard PDF
score_docs = chunk_scorecard(raw_texts["scorecard"])
print(f"Scorecard chunks: {len(score_docs)}")
print(score_docs[1].page_content[:300], "\n--")

Scorecard chunks: 12
300 to 850.  
 
Credit bureau scores consider five general groups of predictive variables: 
– Previous performance, including the severity and frequency of poor performance and 
how recently the poor performance occurred. 
– Current level and use of nonmortgage debt. 
– Amount of time that credit ha 
--


### Combine all chunks

In [51]:
all_docs: List[Document] = deling_docs + fraud_docs + regulatory_docs + score_docs
print(f"Total chunks: {len(all_docs)}")
print(f"By type: {pd.Series([d.metadata['doc_type'] for d in all_docs]).value_counts().to_dict()}")

Total chunks: 105
By type: {'regulatory': 53, 'fraud': 36, 'scorecard': 12, 'delinquency': 4}


## 🧩 Customer-Row → Narrative Document

### Build feature groups

In [52]:
# Fraud-oriented grouping of the SAME 47 columns already in df_enriched.
# Every column name here must exist in df_enriched.columns.

FRAUD_FEATURE_GROUPS = {
    "delinquency_history": [
        "max_recent_level_of_deliq",
        "max_deliq_12mts",
        "max_deliq_6mts",
        "recent_level_of_deliq",
        "max_delinquency_level",
        "num_times_delinquent",
        "time_since_first_deliquency",
        "time_since_recent_deliquency",
        "num_times_60p_dpd",
    ],
    "credit_utilisation": [
        "PL_utilization",
        "CC_utilization",
        "pct_currentBal_all_TL",
        "pct_opened_TLs_L6m_of_L12m",
        "max_unsec_exposure_inPct",
    ],
    "trade_line_activity": [
        "Total_TL_opened_L6M",
        "Tot_TL_closed_L6M",
        "Tot_TL_closed_L12M",
        "pct_tl_open_L6M",
        "pct_tl_closed_L6M",
        "pct_tl_open_L12M",
        "pct_tl_closed_L12M",
    ],
    "product_flags": [
        "PL_Flag", "CC_Flag", "HL_Flag", "GL_Flag",
        "first_prod_enq2", "last_prod_enq2",
    ],
    "account_history": [
        "num_std", "num_std_6mts", "num_std_12mts",
        "num_sub", "num_sub_6mts", "num_sub_12mts",
        "num_lss", "num_lss_6mts", "num_lss_12mts",
        "num_dbt", "num_dbt_6mts", "num_dbt_12mts",
    ],
    "enquiry_pattern": [
        "tot_enq",
    ],
    "payment_behaviour": [
        "Tot_Missed_Pmnt",
        "time_since_recent_payment",
    ],
    "identity_and_profile": [
        "AGE", "GENDER", "EDUCATION", "MARITALSTATUS",
        "NETMONTHLYINCOME",
        "Time_With_Curr_Empr",
        "Credit_Score",
    ],
}

### Derived features

In [53]:
def add_derived_features(row: pd.DataFrame) -> pd.Series:
    """Add derived features to a row based on existing features."""
    r = row.copy()
    def div(a, b):
        return float(a) / float(b) if pd.notna(a) and pd.notna(b) and b not in (0, None) else 0.0

    r["delinq_velocity"]   = div(row.get("num_deliq_6mts", 0), (row.get("num_deliq_12mts", 0) or 0) + 1)
    r["enq_velocity"]      = div(row.get("enq_L3m", 0), (row.get("enq_L12m", 0) or 0) + 1)
    r["unsecured_ratio"]   = div(row.get("Unsecured_TL", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["active_ratio"]      = div(row.get("Tot_Active_TL", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["recent_open_ratio"] = div(row.get("Total_TL_opened_L6M", 0), (row.get("Total_TL", 0) or 0) + 1)
    r["dpd_score"]         = float(row.get("num_times_30p_dpd", 0) or 0) * 1 + float(row.get("num_times_60p_dpd", 0) or 0) * 2
    r["asset_class_score"] = (
        float(row.get("num_sub", 0) or 0) * 1
        + float(row.get("num_dbt", 0) or 0) * 2
        + float(row.get("num_lss", 0) or 0) * 3
    )
    return r

df_enriched = df.apply(add_derived_features, axis=1)
print("\nAdded derived features to the dataset.")
df_enriched[["delinq_velocity","enq_velocity","unsecured_ratio","dpd_score","asset_class_score", "active_ratio", "recent_open_ratio"]].head(3)


Added derived features to the dataset.


,delinq_velocity,enq_velocity,unsecured_ratio,dpd_score,asset_class_score,active_ratio,recent_open_ratio
0,0.0,0.0,0.666667,0.0,0.0,0.166667,0.000000
1,0.0,0.0,0.500000,0.0,0.0,0.500000,0.000000
2,0.1,0.0,0.666667,0.0,0.0,0.888889,0.111111


## Convert each row to a narrative text chunk

In [54]:
def row_to_narrative(row: pd.Series, customer_id: Any = None) -> str:
    """Convert a customer row to a narrative string."""
    parts = [f"FRAUD_REVIEW customer_id={customer_id}"]

    for group, cols in FRAUD_FEATURE_GROUPS.items():
        lines = [f" {c}={row[c]}" for c in cols if c in row.index and pd.notna(row[c])]
        if lines:
            parts.append(f"[{group.upper()}]\n" + "\n".join(lines))

    return "\n".join(parts)

In [55]:
# Customer row -> Langchain Document
id_col = "customer_id" if "customer_id" in df_enriched.columns else None
label_col = "is_fraud" if "is_fraud" in df_enriched.columns else None

# Build function customer doc
def make_customer_doc(row: pd.Series) -> Document:
    cid = row[id_col] if id_col else row.name
    label = int(row[label_col]) if label_col and pd.notna(row[label_col]) else None
    return Document(
        page_content=row_to_narrative(row, cid),
        metadata={"doc_type": "customer_profile", "customer_id": str(cid), "label": label}
    )

# Check usage
print(make_customer_doc(df_enriched.iloc[0]).page_content[:600])
print(make_customer_doc(df_enriched.iloc[0]).metadata)

FRAUD_REVIEW customer_id=0
[DELINQUENCY_HISTORY]
 max_recent_level_of_deliq=29
 max_deliq_12mts=-99999
 max_deliq_6mts=-99999
 recent_level_of_deliq=29
 max_delinquency_level=29
 num_times_delinquent=11
 time_since_first_deliquency=35
 time_since_recent_deliquency=15
 num_times_60p_dpd=0
[CREDIT_UTILISATION]
 PL_utilization=0.798
 CC_utilization=-99999.0
 pct_currentBal_all_TL=0.798
 pct_opened_TLs_L6m_of_L12m=0.0
 max_unsec_exposure_inPct=13.333
[TRADE_LINE_ACTIVITY]
 Total_TL_opened_L6M=0
 Tot_TL_closed_L6M=0
 Tot_TL_closed_L12M=0
 pct_tl_open_L6M=0.0
 pct_tl_closed_L6M=0.0
 pct_tl_open_L12M
{'doc_type': 'customer_profile', 'customer_id': '0', 'label': None}


In [56]:
# Build customer documents
SAMPLE_N = 3000
df_sample = df_enriched.sample(n=SAMPLE_N, random_state=42)

customer_docs = [make_customer_doc(r) for _, r in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Building customer docs")]
print(f"Customer docs sample: {len(customer_docs)}")
print("Customer doc sample:")
print(customer_docs[:1])

Building customer docs:   0%|          | 0/3000 [00:00<?, ?it/s]

Customer docs sample: 3000
Customer doc sample:
[Document(metadata={'doc_type': 'customer_profile', 'customer_id': '8564', 'label': None}, page_content='FRAUD_REVIEW customer_id=8564\n[DELINQUENCY_HISTORY]\n max_recent_level_of_deliq=26\n max_deliq_12mts=0\n max_deliq_6mts=0\n recent_level_of_deliq=26\n max_delinquency_level=26\n num_times_delinquent=2\n time_since_first_deliquency=14\n time_since_recent_deliquency=12\n num_times_60p_dpd=0\n[CREDIT_UTILISATION]\n PL_utilization=-99999.0\n CC_utilization=-99999.0\n pct_currentBal_all_TL=0.039\n pct_opened_TLs_L6m_of_L12m=0.0\n max_unsec_exposure_inPct=2.0\n[TRADE_LINE_ACTIVITY]\n Total_TL_opened_L6M=0\n Tot_TL_closed_L6M=0\n Tot_TL_closed_L12M=0\n pct_tl_open_L6M=0.0\n pct_tl_closed_L6M=0.0\n pct_tl_open_L12M=0.0\n pct_tl_closed_L12M=0.0\n[PRODUCT_FLAGS]\n PL_Flag=0\n CC_Flag=0\n HL_Flag=0\n GL_Flag=0\n first_prod_enq2=others\n last_prod_enq2=ConsumerLoan\n[ACCOUNT_HISTORY]\n num_std=0\n num_std_6mts=0\n num_std_12mts=0\n num_sub=0\n nu

## Vector Store (ChromaDB + HuggingFace Embeddings)

In [57]:
EMBED_MODEL = "all-MiniLM-L6-v2" 

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True}
)
print(f"Embeddings model: {EMBED_MODEL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model: all-MiniLM-L6-v2


## Persist collection to disk for ChromaDB

In [ ]:
import chromadb
import shutil

# 1. Target the exact intended directory safely
PROJECT_ROOT = Path.cwd()
PERSIST_DIR = PROJECT_ROOT / "credit_risk" / "data_science" / "database" / "LLM" / "chroma_store"

# Print to double check the exact physical folder being touched
print(f"Chroma Store Path: {PERSIST_DIR.resolve()}")

# 2. Delete corrupt/legacy SQLite database directory if it exists
if PERSIST_DIR.exists():
    shutil.rmtree(PERSIST_DIR)

PERSIST_DIR.mkdir(parents=True, exist_ok=True)

# 3. Initialize PersistentClient explicitly
client = chromadb.PersistentClient(path=str(PERSIST_DIR))
print("✅ ChromaDB PersistentClient initialized cleanly!")

def build_load_collection(name: str, docs: List[Document]):
    """Store chromadb collection"""
    store = Chroma(
        client=client,
        collection_name=name,
        embedding_function=embeddings
    )
    existing = store._collection.count()

    if existing == 0 and docs:
        BATCH = 256 
        for i in range(0, len(docs), BATCH):
            store.add_documents(docs[i:i+BATCH])

    print(f"Collection '{name}': {store._collection.count()} vectors")
    return store

# Collection (customers + policies brief)
customer_store = build_load_collection("customer_profiles", customer_docs)
policy_docs = deling_docs + fraud_docs + regulatory_docs + score_docs
policy_store = build_load_collection("policy_documents", policy_docs)

print(f"Customer store: {customer_store._collection.count()} vectors")
print(f"Policy store: {policy_store._collection.count()} vectors")

Chroma Store Path: /Users/miftahhadiyannoor/Documents/credit_risk/data_science/notebooks/credit_risk/data_science/database/LLM/chroma_store
✅ ChromaDB PersistentClient initialized cleanly!
Collection 'customer_profiles': 3000 vectors
Collection 'policy_documents': 105 vectors
Customer store: 3000 vectors
Policy store: 105 vectors


## 🤖 LLM Client with HF → Groq Fallback

In [79]:
from huggingface_hub import login
login(token=HUGGINGFACE_API_KEY)
print("✅ Logged in to Hugging Face Hub successfully.")

✅ Logged in to Hugging Face Hub successfully.


In [111]:
# Initialize HF and Groq clients
hf_client = InferenceClient(api_key=HUGGINGFACE_API_KEY) if HUGGINGFACE_API_KEY else None
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def llm_chat(messages: List[Dict[str, str]], max_tokens: int = 256, temperature: float = 0.3) -> Dict[str, Any]:
    """Try HF first, fallback to Groq if HF fails."""
    if hf_client:
        try:
            resp = hf_client.chat.completions.create(
                model="Qwen/Qwen2.5-Coder-32B-Instruct",
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return {"text": resp.choices[0].message.content,
                    "provider": "huggingface"}
        except Exception as e:
            err_msg = str(e)
            if "402" in err_msg or "depleted" in err_msg:
                print("⚠️  HF quota depleted (402). Falling back to Groq...")
            else:
                print(f"❌  Hugging Face failed: {str(e)}")

    # Fallback to Groq
    if groq_client:
        try:
            resp = groq_client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return {"text": resp.choices[0].message.content,
                    "provider": "groq"}
        except Exception as e:
            print(f"❌  Groq failed: {str(e)}")

    raise RuntimeError("⚠️ No LLM provider available. Please check your API keys.")

# Sanity test
print(llm_chat([
    {"role": "user",
     "content": "Reply with the single word: READY"}
]))

{'text': 'READY', 'provider': 'huggingface'}


### Load each model bundles

In [121]:
label_encoders = model_bundle['label_encoders']
scaler = model_bundle['scaler']
models = model_bundle['models']
print(f"Loaded label encoders: {list(label_encoders.keys())}")
print(f"Loaded scaler: {scaler}")
print(f"Loaded models: {list(models.keys())}")

Loaded label encoders: ['first_prod_enq2', 'EDUCATION', 'last_prod_enq2', 'MARITALSTATUS', 'GENDER']
Loaded scaler: StandardScaler()
Loaded models: ['Logistic Regression', 'Random Forest', 'Decision Tree', 'XGBoost', 'K-Nearest Neighbors']


## Scoring ML models return feature with SHAP values

In [ ]:
import shap

def score_row(row: pd.Series, model_name: str) -> dict:
    """Score a single row using the specified model"""
    X = pd.DataFrame([row.reindex(FEATURES).values], columns=FEATURES)

    for col, le in label_encoders.items():
        if col in X.columns:
            X[col] = X[col].astype(str).map(
                lambda s: le.transform([s])[0] if s in le.classes_ else -1
            )

    X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
    X_scaled = pd.DataFrame(scaler.transform(X), columns=FEATURES)

    # Score using the specified model
    model = models[model_name]
    proba = model.predict_proba(X_scaled)[0]
    pred = int(model.predict(X_scaled)[0])
    classes = list(model.classes_)
    fraud_idx = classes.index(1) if 1 in classes else 0

     # Apply SHAP for feature importance
    top_features = []
    try:
        explainer = shap.Explainer(model)
        sv = explainer.shap_values(X_scaled)

        # For binary classification, pick class 1 (fraud class) SHAP values
        if len(sv.shape) == 3:
            row_shap = sv.values[0, :, fraud_idx]
        elif len(sv.shape) == 2:
            row_shap = sv.values[0, :]
        else:
            row_shap = np.array(sv[0])

        # Get original raw feature values for display
        raw_values = row.reindex(FEATURES).to_dict()

        # Build feature list sorted by absolute SHAP contribution
        feature_impacts = [
            {
                "name": feature_name,
                "value": raw_values.get(feature_name, None),
                "contribution": float(shap_val),
                "abs_impact": abs(float(shap_val))
            }
            for feature_name, shap_val in zip(FEATURES, row_shap)
        ]

        # Sort by impact and keep top 5
        sorted_features = sorted(feature_impacts, key=lambda x: x["abs_impact"], reverse=True)

        top_features = [
            {
                "name": f["name"],
                "value": f["value"],
                "contribution": f["contribution"]
            }
            for f in sorted_features[:5]
        ]

    except Exception as e:
        # Fallback to feature_importances_ or coef_ is SHAP fails
        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
            top_idx = np.argsort(importances)[::-1][:5]
            top_features = [
                {
                    "name": FEATURES[i],
                    "value": row[FEATURES[i]],
                    "contribution": float(importances[i])
                }
                for i in top_idx
            ]

    return {
        "fraud_probability": float(proba[fraud_idx]),
        "predicted_label": pred,
        "model_used": model_name,
        "top_features": top_features
    }

## 🧠 RAG Pipeline

In [81]:
def retrieve_context(row: pd.Series, k_customers: int = 3, k_policies: int = 5) -> Dict[str, Any]:
    """Retrieve relevant policy and similar customer documents for a given row"""
    query = row_to_narrative(row)
    cust_hits = customer_store.similarity_search(query, k=k_customers)

    # Policy hits: bias query toward delinquency + enquiries since driver risk
    policy_query = (
        f"{query}\n"
        f"focus: delinquency, {row.get('num_deliq_12mts', 0)} misses in 12 months"
        f"unsecured ratio {row.get('unsecured_ratio', 0):.2f}"
        f"enquiries in last 12 months: {row.get('enq_L12m', 0)}"
        f"asset class score: {row.get('asset_class_score', 0):.2f}"
        f"credit card enquiries: {row.get('CC_enq_L12m', 0)}"
    )
    policy_hits = policy_store.similarity_search(policy_query, k=k_policies)

    return {"customers": cust_hits, "policies": policy_hits}

## Build prompt (system + context + task)

In [82]:
FRAUD_SYSTEM_PROMPT = """You are a senior fraud risk analyst assistant for a bank/fintech.

The ML fraud model has ALREADY produced a fraud probability and a binary label
(1 = fraud, 0 = no fraud). You DO NOT compute, adjust, or override that score.

Your job is to:
    1. Interpret the fraud probability in plain language for a fraud ops analyst.
    2. Identify the single most likely fraud typology the case resembles, using ONLY the typology definitions
    present in the RETRIEVED POLICIES.
    3. List the concrete red flags observed in the CUSTOMER FEATURES. For each red flag, name the feature and quote its value.
    4. Map the case to the retrieved policies and cite them by their exact reference string
    (e.g. "Fraud_Typologies_and_Red Flags.pdf p12").
    5. Recommend exactly ONE action from the ALLOWED ACTIONS list.
    6. State your confidence (0.0-1.0) and, if below 0.6, set needs_human_review=True.

Had rules:
    - NEVER ivent numbers. Use only values that appear in the ML FRAUD OUTPUT
      or CUSTOMER FEATURES blocks.
    - NEVER invent a typology. Pick "unknown" if none of the retrieved typology
      definitions match.
    - NEVER invent a policy citation. Every entry in policy_refs MUST appear
      verbatim in the RETRIEVED POLICIES block.
    - If a required input is missing, write "not available" and explain briefly
      in key_drivers.
    - Prefer specificity: name features and values, not generic phrases.

You must return STRICT JSON only. No prose outside JSON. No markdown fences.
"""

# Constants the prompt ferences
ALLOWED_TYPOLOGIES = [
    "application_fraud",
    "account_takeover",
    "synthetic_identity",
    "first_party_fraud",
    "friendly_fraud",
    "money_mule",
    "card_not_present",
    "unknown",
]

ALLOWED_ACTIONS = [
    "allow",
    "step_up_auth",
    "manual_review",
    "block",
    "report_sar",
]

REQUIRED_KEYS = {
    "fraud_band",
    "typology",
    "red_flags",
    "key_drivers",
    "recommended_action",
    "policy_refs",
    "confidence",
    "needs_human_review",
}

FRAUD_THRESHOLDS = {
    "auto_allow_below": 0.10,
    "auto_block_above": 0.85,
    "k_customers": 5,
    "k_policies": 5,
}

## JSON Parsing

In [ ]:
import json
import re

def parse_fraud_output(text: str) -> dict:
    txt = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()

    try:
        out = json.loads(txt)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", txt, flags=re.DOTALL)
        blob = m.group(0) if m else "{}"
        try:
            out = json.loads(blob)
        except json.JSONDecodeError:
            # one repair pass via the LLM
            repair = llm_chat(
                messages=[
                    {"role": "system",
                     "content": "Fix malformed JSON. Return ONLY valid JSON."},
                    {"role": "user", "content": blob},
                ],
                max_tokens=600,
                temperature=0.0,
            )
            try:
                out = json.loads(repair["text"])
            except json.JSONDecodeError:
                out = {"raw_text": txt, "parse_error": "unrecoverable"}

    if out.get("typology") not in ALLOWED_TYPOLOGIES:
        out["typology"] = "unknown"
    if out.get("recommended_action") not in ALLOWED_ACTIONS:
        out["recommended_action"] = "manual_review"
    if out.get("recommended_action") in {"block", "report_sar"}:   # set, not dict
        out["needs_human_review"] = True

    missing = REQUIRED_KEYS - out.keys()
    if missing:
        out["missing_keys"] = sorted(missing)

    return out

parsed = parse_fraud_output(out["text"])
print(json.dumps(parsed, indent=2))

## 🧪 Call the LLM

In [113]:
out = llm_chat(messages=messages, max_tokens=256, temperature=0.3)

print(f"Provider: {out['provider']}")
print("=" * 60)
print(out['text'])
print("=" * 60)

parsed = parse_fraud_output(out['text'])
print("Parsed output:")
print(json.dumps(parsed, indent=2))

Provider: huggingface
{
  "fraud_band": "Critical",
  "typology": "unknown",
  "red_flags": [
    {"feature": "max_recent_level_of_deliq", "value": "29", "why": "High delinquency level can indicate risky behavior."},
    {"feature": "recent_level_of_deliq", "value": "29", "why": "Recent high delinquency level is a strong indicator of potential fraud."},
    {"feature": "max_delinquency_level", "value": "29", "why": "Maximum delinquency level suggests severe payment issues."},
    {"feature": "num_times_delinquent", "value": "11", "why": "Multiple instances of delinquency are concerning."},
    {"feature": "time_since_first_deliquency", "value": "35", "why": "Short time since first delinquency could indicate ongoing risky behavior."},
    {"feature": "time_since_recent_deliquency", "value": "15", "why": "Very recent delinquency is a significant red flag."},
    {"feature": "PL_utilization", "value": "0.798", "why": "High utilization rate on personal
Parsed output:
{
  "fraud_band": "Cri